# AI013 — Forecasting Test & Evaluation
**Phoenix Project | AI-ML Team**

Loads the trained checkpoint, evaluates on the test set, and generates future forecasts.

In [ ]:
import os, sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from dataset_builder import build_dataset, FEATURE_COLS, TARGET_COL, DATE_COL
from sequence_generator import prepare_sequences
from predictions.predictor import load_model, predict_test, forecast_future, build_future_dataframe
from evaluation.evaluate import (
    set_seeds, compute_metrics, print_metrics, compare_metrics,
    plot_predictions, plot_residuals, plot_metric_comparison, generate_report
)
from validation.validation_label_builder import build_validation_report, label_accuracy

with open('../configs/forecasting.yaml') as f:
    cfg = yaml.safe_load(f)

set_seeds(cfg['training']['random_seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

## Load Data & Checkpoint

In [ ]:
DATA_PATH = os.path.join('..', cfg['dataset']['path'])
df = build_dataset(DATA_PATH, region=cfg['dataset']['region_filter'])

seq = prepare_sequences(
    df,
    feature_cols = FEATURE_COLS,
    target_col   = TARGET_COL,
    window       = cfg['sequence']['window_size'],
    train_split  = cfg['sequence']['train_split'],
    batch_size   = cfg['training']['batch_size']
)

X_test_t      = seq['X_test_t']
y_test_actual = seq['target_scaler'].inverse_transform(seq['y_test_t'].numpy())
target_scaler = seq['target_scaler']
split         = seq['split_idx']

CKPT_PATH = os.path.join('..', cfg['output']['checkpoint_path'])
model, ckpt = load_model(CKPT_PATH, device)
print('model loaded')

## Evaluate on Test Set

In [ ]:
lstm_preds    = predict_test(model, X_test_t, target_scaler, device)
lstm_metrics  = compute_metrics(y_test_actual, lstm_preds)

# rolling mean baseline
from scoring.forecasting_scoring import score_baseline
base_metrics, base_preds = score_baseline(
    seq['y_scaled'], split,
    cfg['sequence']['window_size'],
    len(X_test_t), target_scaler
)

compare_metrics({'LSTM': lstm_metrics, 'Baseline': base_metrics})

In [ ]:
test_dates = df[DATE_COL].iloc[split + cfg['sequence']['window_size'] :].values[:len(y_test_actual)]
plot_predictions(y_test_actual, lstm_preds, base_preds, dates=test_dates)

In [ ]:
plot_residuals(y_test_actual, lstm_preds)

In [ ]:
plot_metric_comparison({'LSTM': lstm_metrics, 'Baseline': base_metrics})

## Validation Labels

In [ ]:
val_report = build_validation_report(y_test_actual, lstm_preds)
print(f'label accuracy: {label_accuracy(y_test_actual, lstm_preds):.2%}')
print(val_report.to_string(index=False))

## Future Forecast (14 Days)

In [ ]:
FUTURE_STEPS = 14
future_preds = forecast_future(
    model,
    seed_window   = seq['X_scaled'][-cfg['sequence']['window_size']:],
    steps         = FUTURE_STEPS,
    target_scaler = target_scaler,
    device        = device
)

future_df = build_future_dataframe(future_preds, df[DATE_COL].max(), freq='D')
print(future_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df[DATE_COL], df[TARGET_COL], color='#2c3e50', linewidth=1.2, label='Historical', alpha=0.6)
ax.plot(future_df['date'], future_df['predicted_frp_mw'],
        color='#e74c3c', linewidth=2, linestyle='--', marker='s', markersize=6, label='Forecast')
ax.axvspan(future_df['date'].iloc[0], future_df['date'].iloc[-1],
           alpha=0.07, color='red', label='Forecast Window')
ax.set_title('Fire Intensity — Historical + 14-Day Forecast', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('FRP (MW)')
ax.legend(); plt.tight_layout(); plt.show()

## Save Report

In [ ]:
os.makedirs('../outputs', exist_ok=True)

results_df = pd.DataFrame({
    'date'        : test_dates,
    'actual'      : y_test_actual.flatten(),
    'predicted'   : lstm_preds.flatten(),
    'baseline'    : base_preds.flatten()
})
results_df.to_csv('../outputs/test_predictions.csv', index=False)

report = generate_report(
    {'LSTM': lstm_metrics, 'Baseline': base_metrics},
    output_path='../outputs/metrics_report.csv'
)
print(report)